# 5) Initial Data Preprocessing & Dataset Documentation

This notebook performs initial data preprocessing and documents the characteristics of the healthcare disease prediction dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

pd.set_option('display.max_columns', None)

In [ ]:
url = 'https://raw.githubusercontent.com/vyasanbmathew2008/Team-6/main/healthcare-disease-prediction/dataset/healthcare_dataset.csv'

df = pd.read_csv(url)

print('Dataset loaded successfully.')

## 1. Initial Dataset Inspection

In [ ]:
print('Dataset Shape:', df.shape)

print('\nColumn Names:')
print(df.columns.tolist())

print('\nFirst 5 Rows:')
display(df.head())

print('\nDataset Information:')
df.info()

## 2. Create Working Copy

In [ ]:
cleaned_df = df.copy()

print('Working copy created.')

## 3. Duplicate Records

In [ ]:
duplicate_count = cleaned_df.duplicated().sum()

print('Duplicate Records:', duplicate_count)

In [ ]:
if duplicate_count > 0:
    cleaned_df = cleaned_df.drop_duplicates().reset_index(drop=True)

print('Shape after duplicate removal:', cleaned_df.shape)

## 4. Missing Values

In [ ]:
missing_values = cleaned_df.isnull().sum()

print('Missing Values:')
display(missing_values[missing_values > 0])

### Handle Missing Target Values

In [ ]:
target_column = 'Medical Condition'

if cleaned_df[target_column].isnull().sum() > 0:
    cleaned_df = cleaned_df.dropna(subset=[target_column])

print('Missing target values:', cleaned_df[target_column].isnull().sum())

### Handle Missing Numerical Values

In [ ]:
numerical_columns = cleaned_df.select_dtypes(include=np.number).columns

for column in numerical_columns:
    if cleaned_df[column].isnull().sum() > 0:
        cleaned_df[column] = cleaned_df[column].fillna(cleaned_df[column].median())

print('Numerical missing values handled.')

### Handle Missing Categorical Values

In [ ]:
categorical_columns = cleaned_df.select_dtypes(include='object').columns

for column in categorical_columns:
    if cleaned_df[column].isnull().sum() > 0:
        cleaned_df[column] = cleaned_df[column].fillna(cleaned_df[column].mode()[0])

print('Categorical missing values handled.')

In [ ]:
remaining_missing = cleaned_df.isnull().sum()

print('Remaining Missing Values:')
display(remaining_missing[remaining_missing > 0])

## 5. Text Cleaning

In [ ]:
text_columns = cleaned_df.select_dtypes(include='object').columns

for column in text_columns:
    cleaned_df[column] = cleaned_df[column].str.replace(r'\\s+', ' ', regex=True).str.strip()

print('Extra spaces removed from text columns.')

## 6. Standardize Patient Names

Patient names in the dataset have inconsistent capitalization. The following preprocessing standardizes the capitalization of each word without removing any words.

In [ ]:
if 'Name' in cleaned_df.columns:
    cleaned_df['Name'] = (
        cleaned_df['Name']
        .str.replace(r'\\s+', ' ', regex=True)
        .str.strip()
        .apply(
            lambda name: ' '.join(word.capitalize() for word in name.split())
            if isinstance(name, str) else name
        )
    )

print('Patient names standardized successfully.')

display(cleaned_df[['Name']].head(10))

### Example

Before preprocessing:

- Bobby JacksOn
- LesLie TErRy
- DaNnY sMitH
- andrEw waTtS

After preprocessing:

- Bobby Jackson
- Leslie Terry
- Danny Smith
- Andrew Watts

## 7. Identify Date Columns

In [ ]:
date_columns = [
    column for column in cleaned_df.columns
    if 'date' in column.lower()
]

print('Date Columns:')
print(date_columns)

## 8. Convert Date Columns

In [ ]:
for column in date_columns:
    cleaned_df[column] = pd.to_datetime(cleaned_df[column], errors='coerce')

print('Date columns converted to datetime format.')

In [ ]:
for column in date_columns:
    print(column, ':', cleaned_df[column].dtype)

## 9. Identify Initially Irrelevant Columns

In [ ]:
columns_to_remove = [
    'Name',
    'Doctor',
    'Hospital',
    'Room Number'
]

columns_to_remove = [
    column for column in columns_to_remove
    if column in cleaned_df.columns
]

print('Columns identified for initial removal:')
print(columns_to_remove)

In [ ]:
cleaned_df = cleaned_df.drop(columns=columns_to_remove)

print('Shape after removing initially irrelevant columns:', cleaned_df.shape)

## 10. Prediction Timing Considerations

Some variables may contain information that becomes available only after diagnosis or treatment. These variables should be evaluated carefully before final model training.

In [ ]:
timing_sensitive_columns = [
    'Discharge Date',
    'Medication',
    'Test Results',
    'Billing Amount'
]

timing_sensitive_columns = [
    column for column in timing_sensitive_columns
    if column in cleaned_df.columns
]

print('Prediction-timing-sensitive columns:')
print(timing_sensitive_columns)

## 11. Create Length of Stay

In [ ]:
if 'Date of Admission' in cleaned_df.columns and 'Discharge Date' in cleaned_df.columns:
    cleaned_df['Length of Stay'] = (
        cleaned_df['Discharge Date'] - cleaned_df['Date of Admission']
    ).dt.days

print('Length of Stay feature created.')

In [ ]:
if 'Length of Stay' in cleaned_df.columns:
    display(cleaned_df['Length of Stay'].describe())

## 12. Validate Target Variable

In [ ]:
target_column = 'Medical Condition'

print('Target Variable:', target_column)

print('\nTarget Categories:')
print(cleaned_df[target_column].unique())

print('\nNumber of Target Classes:')
print(cleaned_df[target_column].nunique())

## 13. Target Distribution

In [ ]:
target_distribution = cleaned_df[target_column].value_counts()

print('Target Distribution:')
display(target_distribution)

In [ ]:
target_percentage = (
    cleaned_df[target_column]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print('Target Distribution (%):')
display(target_percentage)

In [ ]:
plt.figure(figsize=(10, 6))

sns.countplot(
    data=cleaned_df,
    x=target_column,
    order=cleaned_df[target_column].value_counts().index
)

plt.title('Medical Condition Distribution')
plt.xlabel('Medical Condition')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 14. Final Data Quality Checks

In [ ]:
print('Final Duplicate Records:', cleaned_df.duplicated().sum())

final_missing = cleaned_df.isnull().sum()

print('\nFinal Missing Values:')
display(final_missing[final_missing > 0])

print('\nFinal Dataset Shape:', cleaned_df.shape)

In [ ]:
print('Final Dataset Information:')
cleaned_df.info()

In [ ]:
print('First 5 Rows of Preprocessed Dataset:')
display(cleaned_df.head())

## 15. Dataset Characteristics

In [ ]:
print('Number of Rows:', cleaned_df.shape[0])
print('Number of Columns:', cleaned_df.shape[1])

print('\nNumerical Columns:')
print(cleaned_df.select_dtypes(include=np.number).columns.tolist())

print('\nCategorical Columns:')
print(cleaned_df.select_dtypes(include='object').columns.tolist())

print('\nDate Columns:')
print(cleaned_df.select_dtypes(include='datetime').columns.tolist())

In [ ]:
column_summary = pd.DataFrame({
    'Column': cleaned_df.columns,
    'Data Type': cleaned_df.dtypes.astype(str),
    'Unique Values': cleaned_df.nunique(),
    'Missing Values': cleaned_df.isnull().sum()
})

display(column_summary)

## 16. Preprocessing Summary

In [ ]:
print('Initial preprocessing completed.')

print('\nSummary:')
print('- Duplicate records checked and removed.')
print('- Missing target values handled.')
print('- Numerical missing values handled using median.')
print('- Categorical missing values handled using mode.')
print('- Text whitespace standardized.')
print('- Patient name capitalization standardized before removal.')
print('- Date columns converted to datetime.')
print('- Initially irrelevant columns removed.')
print('- Prediction-timing-sensitive variables identified.')
print('- Length of Stay feature created.')
print('- Target variable validated.')

## 17. Create Runtime Directory and Save Initial Preprocessed Data

A dedicated runtime directory is created in the Colab environment to store the initially preprocessed dataset.

In [ ]:
# Create a dedicated directory for runtime-generated files
runtime_dir = '/content/healthcare_runtime'
os.makedirs(runtime_dir, exist_ok=True)

# Save the initial preprocessed dataset
output_path = os.path.join(runtime_dir, 'preprocessed_data.csv')

cleaned_df.to_csv(output_path, index=False)

print(f'Runtime directory created: {runtime_dir}')
print(f'Initial preprocessed dataset saved to: {output_path}')

## 18. Verify Saved Preprocessed Data

In [ ]:
print('Runtime directory:', runtime_dir)
print('File exists:', os.path.exists(output_path))

preprocessed_df = pd.read_csv(output_path)

print('Saved Dataset Shape:', preprocessed_df.shape)

print('\nSaved Dataset Columns:')
print(preprocessed_df.columns.tolist())

print('\nFirst 5 Rows:')
display(preprocessed_df.head())

## 19. Conclusion

The dataset was initially preprocessed by removing duplicate records, handling missing values, standardizing text fields and patient names, converting date columns, removing initially irrelevant columns, and creating the Length of Stay feature.

The target variable, Medical Condition, was validated and its class distribution was examined.

The resulting initially preprocessed dataset was saved as `preprocessed_data.csv` inside the `healthcare_runtime` directory in the Colab runtime.